# Importing Necessary Libraries

In [1]:
import warnings
warnings.filterwarnings("ignore")

In [2]:
!pip install -q transformers datasets accelerate sentencepiece evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 101.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cudf-cu12 2

In [3]:
from transformers import logging

logging.set_verbosity_warning()   # only important warnings

In [4]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import StratifiedKFold, GroupKFold
from lightgbm import LGBMClassifier, LGBMRanker

from scipy.sparse import hstack
from tqdm import tqdm
from scipy.sparse import csr_matrix
from sklearn.metrics.pairwise import cosine_similarity

import torch

from datasets import Dataset

from transformers import (
    AutoTokenizer,
    AutoModelForMultipleChoice,
    TrainingArguments,
    Trainer
)

from sklearn.model_selection import train_test_split

import evaluate

# Data Importing

In [5]:
train_df = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv')
test_df = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv')
submission_df = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv')

# Model 1 

In [6]:
train_data = train_df.copy()
test_data = test_df.copy()
submission_data = submission_df.copy()

In [7]:
print("Dataset preview : \n")
train_data.head()

Dataset preview : 



,id,prompt,A,B,C,D,E,answer
0,1,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
1,2,What is accelerator-based light-ion fusion?,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,A
2,3,Determine the correct option: What is the term...,Blueshifting,Redshifting,Reddening,Whitening,Yellowing,C
3,4,Select the most accurate option: What is Marti...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
4,5,Identify the correct statement: What is the co...,"Simultaneity is relative, meaning that two eve...","Simultaneity is relative, meaning that two eve...","Simultaneity is absolute, meaning that two eve...",Simultaneity is a concept that applies only to...,Simultaneity is a concept that applies only to...,A


In [8]:
print(f"Train data size : {train_data.shape}")
print(f"Test data size : {test_data.shape}")

Train data size : (2000, 8)
Test data size : (500, 7)


## Preprocessing

In [9]:
def clean_text(text):
    text = str(text)
    text = text.lower()
    text = text.replace("\n"," ")
    text = " ".join(text.split())
    return text

In [10]:
for col in ["prompt","A","B","C","D","E"]:
    train_data[col] = train_data[col].fillna("").apply(clean_text)
    test_data[col] = test_data[col].fillna("").apply(clean_text)

## Convert MCQ into Ranking Format

In [11]:
options = ["A","B","C","D","E"]

train_rows = []

for _, row in tqdm(train_data.iterrows(), total=len(train_data)):
    
    correct = row["answer"]
    
    for op in options:
        
        train_rows.append({
            "id": row["id"],
            "prompt": row["prompt"],
            "option": row[op],
            "option_label": op,
            "label": int(op == correct)
        })

train_long = pd.DataFrame(train_rows)

print(train_long.shape)

100%|██████████| 2000/2000 [00:00<00:00, 16917.39it/s]

(10000, 5)


## Test Ranking Format

In [12]:
test_rows = []

for _, row in tqdm(test_data.iterrows(), total=len(test_data)):
    
    for op in options:
        
        test_rows.append({
            "id": row["id"],
            "option_label": op,
            "prompt": row["prompt"],
            "option": row[op]
        })

test_long = pd.DataFrame(test_rows)

100%|██████████| 500/500 [00:00<00:00, 17153.50it/s]


## Create Additional Features

### Length Features

In [13]:
for df in [train_long, test_long]:
    df["prompt_len"] = df["prompt"].str.len()
    df["option_len"] = df["option"].str.len()
    df["prompt_words"] = df["prompt"].str.split().str.len()
    df["option_words"] = df["option"].str.split().str.len()

In [14]:
num_cols = [
    "prompt_len",
    "option_len",
    "prompt_words",
    "option_words"
]

### Word Overlap and Similarity Feature

In [15]:
def overlap_count(q, a):
    
    q_words = set(q.split())
    a_words = set(a.split())
    
    return len(q_words & a_words)
def overlap_ratio(q, a):

    q_words = set(q.split())
    a_words = set(a.split())

    if len(q_words) == 0:
        return 0

    return len(q_words & a_words) / len(q_words)

In [16]:
for df in [train_long, test_long]:

    df["overlap"] = [
        overlap_count(q, a)
        for q, a in zip(df["prompt"], df["option"])
    ]
    df["overlap_ratio"] = [
    overlap_ratio(q, a)
    for q, a in zip(df["prompt"], df["option"])
    ]

## TF-IDF Feature Engineering

In [17]:
train_text = (
    train_long["prompt"] +
    " [SEP] " +
    train_long["option"]
)

test_text = (
    test_long["prompt"] +
    " [SEP] " +
    test_long["option"]
)

In [18]:
tfidf = TfidfVectorizer(
    max_features=50000,
    ngram_range=(1,2),
    min_df=2,
    max_df=0.95,
    sublinear_tf=True,
    stop_words="english"
)

In [19]:
char_vec = TfidfVectorizer(
    analyzer="char_wb",
    ngram_range=(3,6),
    min_df=2,
    max_features=30000
)

In [20]:
X_char_train = char_vec.fit_transform(train_text)
X_char_test = char_vec.transform(test_text)

In [21]:
X_word_train = tfidf.fit_transform(train_text)

X_word_test = tfidf.transform(test_text)

### Add Cosine Similarity

In [22]:
train_prompt_vec = tfidf.transform(train_long["prompt"])
train_option_vec = tfidf.transform(train_long["option"])

test_prompt_vec = tfidf.transform(test_long["prompt"])
test_option_vec = tfidf.transform(test_long["option"])

In [23]:
train_long["cosine_sim"] = [
    cosine_similarity(
        train_prompt_vec[i],
        train_option_vec[i]
    )[0][0]
    for i in range(len(train_long))
]

test_long["cosine_sim"] = [
    cosine_similarity(
        test_prompt_vec[i],
        test_option_vec[i]
    )[0][0]
    for i in range(len(test_long))
]

In [24]:
num_cols.extend([
    "overlap",
    "overlap_ratio",
    "cosine_sim"
])

In [25]:
num_cols

['prompt_len',
 'option_len',
 'prompt_words',
 'option_words',
 'overlap',
 'overlap_ratio',
 'cosine_sim']

In [26]:
question_ids = train_long["id"]

In [27]:
X_num_train = csr_matrix(
    train_long[num_cols].values.astype(np.float32)
)

X_num_test = csr_matrix(
    test_long[num_cols].values.astype(np.float32)
)

In [28]:
X_train = hstack([
    X_word_train,
    X_char_train,
    X_num_train
]).tocsr()

X_test = hstack([
    X_word_test,
    X_char_test,
    X_num_test
]).tocsr()

In [29]:
print(type(X_train))
print(type(X_test))

<class 'scipy.sparse._csr.csr_matrix'>
<class 'scipy.sparse._csr.csr_matrix'>


In [30]:
print(X_word_train.shape)
print(X_char_train.shape)
print(X_num_train.shape)
print(X_train.shape)

(10000, 11244)
(10000, 30000)
(10000, 7)
(10000, 41251)


In [31]:
train_long[num_cols].isnull().sum()

prompt_len       0
option_len       0
prompt_words     0
option_words     0
overlap          0
overlap_ratio    0
cosine_sim       0
dtype: int64

In [32]:
y = train_long["label"]

## LightGBM

### LGBMClassifier

In [33]:
lgb_params = {
    "n_estimators":1500,
    "learning_rate":0.03,
    "num_leaves":127,
    "subsample":0.8,
    "colsample_bytree":0.8,
    "objective":"binary",
    "metric":"auc",
    "verbosity":-1,
    "random_state":42
}

### Train

In [34]:
# # skf = StratifiedKFold(
# #     n_splits=5,
# #     shuffle=True,
# #     random_state=42
# # )
# gkf = GroupKFold(n_splits=5)

# oof = np.zeros(len(train_long))
# test_preds = np.zeros(len(test_long))

# # for fold, (tr_idx, val_idx) in enumerate(
# #         skf.split(X_train, y)):
# groups = train_long["id"]

# for fold, (tr_idx, val_idx) in enumerate(
#     gkf.split(X_train, y, groups)
# ):
    
#     print(f"Fold {fold+1}")
    
#     model = LGBMClassifier(**lgb_params)
    
#     model.fit(
#         X_train[tr_idx],
#         y.iloc[tr_idx]
#     )
    
#     oof[val_idx] = model.predict_proba(
#         X_train[val_idx]
#     )[:,1]
    
#     test_preds += (
#         model.predict_proba(X_test)[:,1] / 5
#     )
#     print(test_preds)

### LGBMRanker

In [35]:
ranker_params = {
    "objective": "lambdarank",
    "metric": "ndcg",
    "n_estimators": 500,
    "learning_rate": 0.05,
    "num_leaves": 63,
    "force_col_wise": True,
    "verbosity": -1,
    "random_state": 42,
    "n_jobs": -1
}

In [36]:
print(X_train.shape)
print(y.value_counts())

(10000, 41251)
label
0    8000
1    2000
Name: count, dtype: int64


### Training

In [37]:
gkf = GroupKFold(n_splits=5)

groups = train_long["id"]

oof = np.zeros(len(train_long))
test_preds = np.zeros(len(test_long))

for fold, (tr_idx, val_idx) in enumerate(
        gkf.split(X_train, y, groups)
):

    print(f"Fold {fold+1}")

    ranker = LGBMRanker(**ranker_params)

    # group sizes for training fold
    train_groups = (
        train_long.iloc[tr_idx]
        .groupby("id")
        .size()
        .values
    )

    valid_groups = (
        train_long.iloc[val_idx]
        .groupby("id")
        .size()
        .values
    )

    ranker.fit(
        X_train[tr_idx],
        y.iloc[tr_idx],
        group=train_groups,
        eval_set=[(X_train[val_idx], y.iloc[val_idx])],
        eval_group=[valid_groups]
    )

    oof[val_idx] = ranker.predict(
        X_train[val_idx]
    )

    test_preds += (
        ranker.predict(X_test) / 5
    )

Fold 1
Fold 2
Fold 3
Fold 4
Fold 5


## Save the artifacts

In [38]:
import joblib

joblib.dump(
    ranker,
    "lgbm_ranker_v1.pkl"
)

joblib.dump(
    tfidf,
    "word_tfidf_v1.pkl"
)

joblib.dump(
    char_vec,
    "char_tfidf_v1.pkl"
)

['char_tfidf_v1.pkl']

## Evaluation

In [39]:
def map3_score(df):

    scores = []

    for _, g in df.groupby("id"):

        pred = (
            g.sort_values(
                "oof_pred",
                ascending=False
            )["option_label"]
            .tolist()[:3]
        )

        true = g.loc[
            g["label"] == 1,
            "option_label"
        ].iloc[0]

        if true == pred[0]:
            scores.append(1.0)

        elif true == pred[1]:
            scores.append(0.5)

        elif true == pred[2]:
            scores.append(1/3)

        else:
            scores.append(0)

    return np.mean(scores)

In [40]:
train_long["oof_pred"] = oof

cv_score = map3_score(train_long)

print("CV MAP@3 =", cv_score)

CV MAP@3 = 0.991


In [41]:
# for _, g in train_long.groupby("id"):
#     print(g[["option_label","label","oof_pred"]])
#     break

In [42]:
correct = 0

for _, g in train_long.groupby("id"):
    pred = g.loc[g["oof_pred"].idxmax(), "option_label"]
    true = g.loc[g["label"] == 1, "option_label"].iloc[0]

    if pred == true:
        correct += 1

print(correct / train_data.shape[0])

0.9855


In [43]:
train_ids = set(train_long.iloc[tr_idx]["id"])
val_ids = set(train_long.iloc[val_idx]["id"])

print(len(train_ids & val_ids))

0


In [44]:
for _, g in train_long.groupby("id"):

    print(
        g[["option_label","label","oof_pred"]]
        .sort_values("oof_pred", ascending=False)
    )

    break

  option_label  label  oof_pred
1            B      1  6.359652
2            C      0 -3.568744
0            A      0 -3.946336
4            E      0 -4.441805
3            D      0 -5.361554


In [45]:
sample_ids = np.random.choice(
    train_long["id"].unique(),
    10,
    replace=False
)

for sid in sample_ids:

    g = train_long[train_long["id"] == sid]

    print("\n", sid)

    print(
        g[["option_label","label","oof_pred"]]
        .sort_values(
            "oof_pred",
            ascending=False
        )
    )


 824
     option_label  label  oof_pred
4118            D      1  4.510287
4117            C      0 -3.282293
4119            E      0 -5.062391
4115            A      0 -5.890268
4116            B      0 -7.700518

 1110
     option_label  label  oof_pred
5545            A      1  2.785802
5548            D      0 -1.987676
5547            C      0 -2.010992
5546            B      0 -3.195338
5549            E      0 -4.318139

 555
     option_label  label  oof_pred
2774            E      1  3.207602
2772            C      0 -3.666378
2771            B      0 -4.655557
2773            D      0 -5.226142
2770            A      0 -6.469153

 215
     option_label  label  oof_pred
1072            C      1 -0.044552
1073            D      0 -3.461734
1071            B      0 -3.461734
1070            A      0 -3.987232
1074            E      0 -6.536217

 160
    option_label  label  oof_pred
795            A      1  4.033766
799            E      0  0.252690
798            D      0 -2.

## Predictions

In [46]:
test_long["score"] = test_preds

In [47]:
test_long.head()

,id,option_label,prompt,option,prompt_len,option_len,prompt_words,option_words,overlap,overlap_ratio,cosine_sim,score
0,1,A,pick the best possible answer: what is the rel...,"for every eigenstate of one hamiltonian, its p...",144,117,19,17,1,0.058824,0.0,4.808152
1,1,B,pick the best possible answer: what is the rel...,"for every eigenstate of one hamiltonian, its p...",144,117,19,17,0,0.000000,0.0,-4.434247
2,1,C,pick the best possible answer: what is the rel...,"for every eigenstate of one hamiltonian, its p...",144,118,19,17,0,0.000000,0.0,-5.037296
3,1,D,pick the best possible answer: what is the rel...,"for every eigenstate of one hamiltonian, its p...",144,120,19,17,0,0.000000,0.0,-4.975927
4,1,E,pick the best possible answer: what is the rel...,"for every eigenstate of one hamiltonian, its p...",144,116,19,17,0,0.000000,0.0,-4.501600


## Submission Data Preparation for Model 1

In [48]:
submission_rows = []

for qid, group in test_long.groupby("id"):
    
    ranked = group.sort_values(
        "score",
        ascending=False
    )
    
    top3 = ranked["option_label"].head(3).tolist()
    
    submission_rows.append({
        "id": qid,
        "prediction": " ".join(top3)
    })

In [49]:
submission = pd.DataFrame(submission_rows)
submission.head()

,id,prediction
0,1,A B E
1,2,B D E
2,3,B D E
3,4,E C D
4,5,C D A


In [50]:
submission.to_csv(
    "submission.csv",
    index=False
)

# Model 2 

In [51]:
train_data = train_df.copy()
test_data = test_df.copy()

print(train_data.shape)
print(test_data.shape)

(2000, 8)
(500, 7)


In [52]:
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

True
Tesla T4


In [53]:
device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "cpu"
)

print(device)

cuda


## Encode Labels

In [54]:
label_map = {
    "A":0,
    "B":1,
    "C":2,
    "D":3,
    "E":4
}

train_data["label"] = train_data["answer"].map(label_map)

## Train-Validation Split

In [55]:
train_part, valid_part = train_test_split(
    train_data,
    test_size=0.2,
    stratify=train_data["label"],
    random_state=42
)

## Convert to Hugging Face Dataset

In [56]:
train_ds = Dataset.from_pandas(train_part)
valid_ds = Dataset.from_pandas(valid_part)

test_ds = Dataset.from_pandas(
    test_data.assign(label=0)
)

## Load DeBERTa Tokenizer

In [57]:
MODEL_NAME = "microsoft/deberta-v3-base"

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)

config.json:   0%|          | 0.00/579 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

In [58]:
MAX_LEN = 256

def preprocess(example):

    first_sentences = [
        example["prompt"]
    ] * 5

    second_sentences = [
        example["A"],
        example["B"],
        example["C"],
        example["D"],
        example["E"]
    ]

    tokenized = tokenizer(
        first_sentences,
        second_sentences,
        truncation=True,
        max_length=MAX_LEN
    )

    tokenized["label"] = example["label"]

    return tokenized

In [59]:
train_ds = train_ds.map(preprocess)
valid_ds = valid_ds.map(preprocess)
test_ds = test_ds.map(preprocess)

Map:   0%|          | 0/1600 [00:00<?, ? examples/s]

Map:   0%|          | 0/400 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

In [60]:
keep_cols = [
    "input_ids",
    "attention_mask",
    "token_type_ids",
    "label"
]

train_ds = train_ds.remove_columns(
    [c for c in train_ds.column_names if c not in keep_cols]
)

valid_ds = valid_ds.remove_columns(
    [c for c in valid_ds.column_names if c not in keep_cols]
)

test_ds = test_ds.remove_columns(
    [c for c in test_ds.column_names if c not in keep_cols]
)

In [61]:
sample = train_ds[0]

print(type(sample["input_ids"]))
print(len(sample["input_ids"]))
print(train_ds.column_names)

for k,v in sample.items():
    print(k, type(v))

<class 'list'>
5
['label', 'input_ids', 'token_type_ids', 'attention_mask']
label <class 'int'>
input_ids <class 'list'>
token_type_ids <class 'list'>
attention_mask <class 'list'>


## Data Collector

In [62]:
from dataclasses import dataclass
from transformers.tokenization_utils_base import PreTrainedTokenizerBase

@dataclass
class DataCollatorForMultipleChoice:

    tokenizer: PreTrainedTokenizerBase

    def __call__(self, features):

        labels = [f.pop("label") for f in features]

        batch_size = len(features)
        num_choices = len(features[0]["input_ids"])

        flattened = [
            [
                {
                    k: v[i]
                    for k, v in feature.items()
                    if k!= 'label'
                }
                for i in range(num_choices)
            ]
            for feature in features
        ]

        flattened = sum(flattened, [])

        batch = self.tokenizer.pad(
            flattened,
            return_tensors="pt"
        )

        batch = {
            k: v.view(
                batch_size,
                num_choices,
                -1
            )
            for k, v in batch.items()
        }
        if "token_type_ids" in batch:
            del batch["token_type_ids"]
        
        batch["labels"] = torch.tensor(labels)

        return batch

## Load Model

In [63]:
model = AutoModelForMultipleChoice.from_pretrained(
    MODEL_NAME,
    dtype = torch.float32
)

pytorch_model.bin:   0%|          | 0.00/371M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/371M [00:00<?, ?B/s]

DebertaV2ForMultipleChoice LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     | 
----------------------------------------+------------+-
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
classifier.weight                       | MISSING    | 
pooler.dense.weight                     | MISSING    | 
classifier.bias                         | MISSING    | 
pooler.dense.bias                

In [64]:
print(next(model.parameters()).dtype)

torch.float32


## Metric

In [65]:
accuracy = evaluate.load("accuracy")

def compute_metrics(eval_pred):

    logits, labels = eval_pred

    preds = np.argmax(
        logits,
        axis=1
    )

    return accuracy.compute(
        predictions=preds,
        references=labels
    )

## Training Arguments

In [66]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="deberta_mcq",

    learning_rate=2e-5,

    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,

    gradient_accumulation_steps=2,

    num_train_epochs=4,

    fp16=False,

    eval_strategy="epoch",

    save_strategy="epoch",

    load_best_model_at_end=True,

    logging_steps=25,
    
    weight_decay = 0.01,

    report_to="none"
)

In [67]:
collator = DataCollatorForMultipleChoice(
    tokenizer
)

batch = collator(
    [train_ds[0], train_ds[1]]
)
batch = {k:v.to(model.device) for k,v in batch.items()}

outputs = model(**batch)

print(outputs.logits.shape)
print(outputs.loss)
print(outputs.logits)

torch.Size([2, 5])
tensor(1.6194, grad_fn=<NllLossBackward0>)
tensor([[-0.1620, -0.1969, -0.1177, -0.1635, -0.0527],
        [-0.0331, -0.1974, -0.1737, -0.1838, -0.1939]],
       grad_fn=<ViewBackward0>)


## Trainer

In [68]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=valid_ds,
    processing_class=tokenizer,
    data_collator=collator,
    compute_metrics=compute_metrics
)

## Train

In [69]:
trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 2, 'bos_token_id': 1}.


Epoch,Training Loss,Validation Loss,Accuracy
1,5.153267,2.053441,0.607500
2,3.443059,1.290585,0.815000
3,3.086430,0.924062,0.890000
4,2.361129,0.847004,0.897500


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['deberta.embeddings.LayerNorm.weight', 'deberta.embeddings.LayerNorm.bias', 'deberta.encoder.layer.0.attention.output.LayerNorm.weight', 'deberta.encoder.layer.0.attention.output.LayerNorm.bias', 'deberta.encoder.layer.0.output.LayerNorm.weight', 'deberta.encoder.layer.0.output.LayerNorm.bias', 'deberta.encoder.layer.1.attention.output.LayerNorm.weight', 'deberta.encoder.layer.1.attention.output.LayerNorm.bias', 'deberta.encoder.layer.1.output.LayerNorm.weight', 'deberta.encoder.layer.1.output.LayerNorm.bias', 'deberta.encoder.layer.2.attention.output.LayerNorm.weight', 'deberta.encoder.layer.2.attention.output.LayerNorm.bias', 'deberta.encoder.layer.2.output.LayerNorm.weight', 'deberta.encoder.layer.2.output.LayerNorm.bias', 'deberta.encoder.layer.3.attention.output.LayerNorm.weight', 'deberta.encoder.layer.3.attention.output.LayerNorm.bias', 'deberta.encoder.layer.3.output.LayerNorm.weight', 'deberta.encoder.layer.3.output.Laye

TrainOutput(global_step=400, training_loss=3.89779390335083, metrics={'train_runtime': 527.4821, 'train_samples_per_second': 12.133, 'train_steps_per_second': 0.758, 'total_flos': 1599359539195680.0, 'train_loss': 3.89779390335083, 'epoch': 4.0})

## Save the artifacts

In [70]:
trainer.save_model(
    "/kaggle/working/model2_deberta_v1"
)

tokenizer.save_pretrained(
    "/kaggle/working/model2_deberta_v1"
)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('/kaggle/working/model2_deberta_v1/tokenizer_config.json',
 '/kaggle/working/model2_deberta_v1/tokenizer.json')

## Predict Test Logic

In [71]:
preds = trainer.predict(test_ds)

logits = preds.predictions

In [72]:
np.save(
    "/kaggle/working/deberta_logits_v1.npy",
    logits
)

## Convert Logits to Top-3

In [73]:
labels = np.array(
    ["A","B","C","D","E"]
)

top3_predictions = []

for row in logits:

    idx = np.argsort(row)[::-1][:3]

    top3_predictions.append(
        " ".join(labels[idx])
    )

## Submission for Model 2

In [74]:
submission = pd.DataFrame({
    "id": test_df["id"],
    "prediction": top3_predictions
})

submission.to_csv(
    "submission.csv",
    index=False
)

submission.head()

,id,prediction
0,1,A D B
1,2,B C D
2,3,B E D
3,4,E C A
4,5,C D A
